# Calendar Agent — Colab LoRA 학습 (git 방식 · 단일 GPU · bf16)

캐글 노트북과 동일한 git 흐름의 Colab판. **HF 불필요** — 결과는 zip으로 PC에 다운로드.

**준비**
1. 상단 `런타임 → 런타임 유형 변경 → T4 GPU`
2. GitHub PAT (repo scope): https://github.com/settings/tokens/new?type=classic

셀을 위→아래 실행. 단일 T4 기준 0.5B LoRA ~1~1.5h. 인증은 GitHub PAT 1회뿐.

In [ ]:
# 0. GPU 확인 — Tesla T4 떠야 함. 안 뜨면 런타임 유형을 GPU로.
!nvidia-smi

In [ ]:
# 1. repo clone (private → PAT). 이미 있으면 origin/main으로 force-sync
import os, getpass
%cd /content
if not os.path.exists('calendar-agent'):
    token = getpass.getpass('GitHub PAT (repo scope): ').strip()
    !git clone https://{token}@github.com/sooryong/calendar-agent.git
    token = None
else:
    !cd calendar-agent && git fetch origin && git reset --hard origin/main
%cd /content/calendar-agent
!git log --oneline -1

In [ ]:
# 2. 설치 + 환경 (wandb off). torch가 +cpu면 GPU 미할당 → 런타임 GPU로 바꾸고 재시작
!pip install -q -e .[train]
!pip uninstall torchao -y -q   # PEFT가 거부하는 구버전 torchao 제거 (LoRA엔 불필요)
import os, torch
os.environ['WANDB_DISABLED'] = 'true'
print(f"torch={torch.__version__}  cuda={torch.cuda.is_available()}  "
      f"device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
assert torch.cuda.is_available(), 'GPU 미할당 — 런타임을 T4 GPU로 바꾸고 셀 재실행'

In [ ]:
# 3. 데이터 확인 (repo에 포함 — clone으로 따라옴). golden 50건이어야 현 기준 평가
for p in ['data/processed/train.jsonl', 'data/processed/val.jsonl', 'data/eval/real_golden.jsonl']:
    n = sum(1 for l in open(p, encoding='utf-8') if l.strip())
    print(f'{p}: {n}')
n_gold = sum(1 for l in open('data/eval/real_golden.jsonl', encoding='utf-8') if l.strip())
assert n_gold == 50, f'golden {n_gold}건 (50 기대) — origin/main 동기화 확인'

In [ ]:
# 4. 학습 (단일 GPU · bf16 · configs/train.yaml = r27). torchrun 아님(캐글은 2-GPU torchrun).
#    시작 로그에 '[train] 모델: Qwen/Qwen2.5-0.5B-Instruct' 확인. ~1~1.5h
!python scripts/train_lora.py --config configs/train.yaml

In [ ]:
# 5. merge + 골든 평가 (train.yaml output_dir에서 라운드 자동 인식)
import yaml, os
cfg = yaml.safe_load(open('configs/train.yaml'))
LORA_DIR = cfg['output_dir']                 # models/lora/r27-qwen0.5b
NAME     = os.path.basename(LORA_DIR)        # r27-qwen0.5b
MERGED   = f'models/merged/{NAME}'
EVAL_JSON= f'logs/eval_{NAME}.json'
print(f'round name={NAME}')
!python scripts/merge_lora.py --base Qwen/Qwen2.5-0.5B-Instruct --lora {LORA_DIR} --out {MERGED}
!python scripts/eval_model.py --model {MERGED} --eval data/eval/real_golden.jsonl --out {EVAL_JSON}
print('--- ' + EVAL_JSON + ' ---')
print(open(EVAL_JSON, encoding='utf-8').read())

In [ ]:
# 6. 어댑터 zip → PC로 다운로드 (세션 정리 전 꼭 받기). 캐글 cell 6과 동일.
ZIP = f'/content/lora_{NAME}.zip'
!zip -qr {ZIP} {LORA_DIR} {EVAL_JSON} configs/train.yaml configs/lora.yaml configs/model_qwen.yaml
!ls -la {ZIP}
from google.colab import files
files.download(ZIP)